In [ ]:
import inspect

In [ ]:

############################################
# Smart Instance: Instantiator Class
############################################

class Instantiator:    
    def __init__(self, script: str, block_hash:str, smart_references:dict=None):
        self.smarthash = block_hash
        self.smart_references = smart_references
        self.namespace = {"SIEvent":self.SIEvent}
        self.smart_importer()
        exec(script, self.namespace)
        self.classes = {name: obj for name, obj in self.namespace.items() if isinstance(obj, type)}

    def SIEvent(self,data):
        transaction = {
            'EventMetadata': {
                'timestamp': datetime.now().timestamp(),
                'smarthash': self.smarthash,
            },
            'body': data
        }
        return transaction

    def smart_importer(self):
        for smarthash,script in self.smart_references.items():
            exec(script,self.namespace)

    
    def get_class_names(self):
        return list(self.classes.keys())
    
    def instantiate(self, class_name: str, *args, **kwargs):
        if class_name not in self.classes:
            raise ValueError(f"Class '{class_name}' not found in the provided script.")
        return self.classes[class_name](*args, **kwargs)
    
    def list_members(self, class_name: str):
        if class_name not in self.classes:
            raise ValueError(f"Class '{class_name}' not found in the provided script.")
        
        cls = self.classes[class_name]
        member_functions = [
            name for name, func in inspect.getmembers(cls, predicate=inspect.isfunction)
            if not name.startswith('__') and not name.startswith('_') and not name.startswith('SIEvent')
        ]
        
        try:
            sig = inspect.signature(cls.__init__)
            dummy_args = {}
            for param_name, param in sig.parameters.items():
                if param_name == 'self':
                    continue
                dummy_args[param_name] = param.default if param.default is not param.empty else f"dummy_{param_name}"
            instance = cls(**dummy_args)
            member_variables = list(vars(instance).keys())
        except Exception as e:
            member_variables = f"Could not instantiate class to list member variables: {e}"
        
        return member_variables, member_functions
    
    def run_member_function(self, instance, function_name: str, *args, **kwargs):
        if hasattr(instance, function_name):
            method = getattr(instance, function_name)
            return method(*args, **kwargs)
        else:
            raise AttributeError(f"Method '{function_name}' not found in instance of {type(instance).__name__}")
    
    def run_all_member_functions(self, instance, functions_params: dict = None):
        outputs = {}
        member_functions = [
            name for name, func in inspect.getmembers(instance, predicate=inspect.ismethod)
            if not name.startswith('__') and not name.startswith('_')
        ]
        for func_name in member_functions:
            args, kwargs = (), {}
            if functions_params and func_name in functions_params:
                args, kwargs = functions_params[func_name]
            try:
                outputs[func_name] = self.run_member_function(instance, func_name, *args, **kwargs)
            except Exception as e:
                outputs[func_name] = f"Error: {e}"
        return outputs
    
    def instantiate_composite(self, *args, **kwargs):
        if not self.classes:
            raise ValueError("No classes loaded from the provided script.")
        
        def get_depth(cls):
            depth = 0
            for base in cls.__mro__:
                if base is object:
                    break
                depth += 1
            return depth
        
        most_derived = max(self.classes.values(), key=get_depth)
        mro_set = set(most_derived.__mro__)
        other_bases = tuple(cls for cls in self.classes.values() 
                            if cls is not most_derived and cls not in mro_set)
        
        if not other_bases:
            Composite = most_derived
        else:
            bases = (most_derived,) + other_bases
            Composite = type("Composite", bases, {})
        
        return Composite(*args, **kwargs)
    
    def get_composite_init_signature(self):
        """
        Returns the signature of the __init__ method used by the composite instantiator.
        """
        if not self.classes:
            raise ValueError("No classes loaded from the provided script.")
        
        def get_depth(cls):
            depth = 0
            for base in cls.__mro__:
                if base is object:
                    break
                depth += 1
            return depth
        
        most_derived = max(self.classes.values(), key=get_depth)
        return inspect.signature(most_derived.__init__)
    
    def instantiate_composite_with_params(self, params: dict=None):
        """
        Instantiates the composite class using parameters from the provided dictionary.
        
        Raises:
            ValueError: If a required parameter is missing.
        """
        sig = self.get_composite_init_signature()
        required_params = [name for name, param in sig.parameters.items() if name != "self"]
        
        for param in required_params:
            if param not in params:
                raise ValueError(f"Missing required parameter: {param}")
        
        return self.instantiate_composite(**params)